In [2]:
import re
from pprint import pprint 
import pandas as pd 
import wptools
import json
from tqdm import tqdm
from refined.inference.processor import Refined

refined = Refined.from_pretrained(model_name='wikipedia_model_with_numbers',
                                  entity_set="wikipedia")
folder = "/Users/amycweng/My Drive (aw3029@princeton.edu)/DH/POETRY/APS"

In [ ]:
authors = pd.read_csv(f"{folder}/APS_authors.csv")['name']
linked_authors = []
for aut in tqdm(authors): 
    spans = refined.process_text(aut)
    for s in spans: 
        ent_type = s.coarse_mention_type
        if ent_type != "PERSON": continue # only humans allowed 
        ent = s.predicted_entity
        if ent is not None: 
            linked_authors.append((aut,ent_type,ent.wikipedia_entity_title,ent.wikidata_entity_id))
        else: 
            linked_authors.append((aut,ent_type,'',''))
with open(f"{folder}/APS_authors_linked.json","w+") as file: 
    json.dump(linked_authors,file)

# Proper Nouns and Italicized Phrases in GROUND corpus 

In [10]:
spans_df = pd.read_csv(f"{folder}/features/ground_spans.csv")
spans_df = spans_df.fillna("")
spans_df = spans_df[spans_df['tag'].str.contains("np|<i>")]
spans_df = spans_df[spans_df['pos'].str.contains("np")]
spans_df 

,tag,frequency,tokens,pos,lemmata
0,np,85332,god,np1,god
1,np,34257,christ,np1,christ
2,np,10269,gods,npg1,god
3,np,5913,cor.,np1,cor.
4,np,5848,rom.,np1,rom.
...,...,...,...,...,...
282276,<i>,1,"caesar , qui apud te ,","np1 , fw-la fw-la fw-la ,","caesar , qui apud te ,"
282286,<i>,1,"priests , moses , aaron , dauid , salomon , sa...","n2 , np1 , np1 , np1 , np1 , np1 ,","priest , moses , aaron , david , salomon , sam..."
282291,<i>,1,"bernard , quanta cum humilitate debet rana pau...","np1 , fw-la fw-la fw-la fw-la fw-la fw-la fw-l...","bernard , quanta cum humilitate debet rana pau..."
282308,<i>,1,"god forgiue vs ,","np1 vvb pno12 ,","god forgive we ,"


In [16]:
from abbreviations import abbrev_to_book
# Bible book abbreviations 
len(abbrev_to_book)

1705

In [25]:
spans_dict = spans_df.to_dict(orient='records')
linked = {}
for idx, entry in tqdm(enumerate(spans_dict)): 
    item = entry['lemmata'].capitalize()
    if item[-1] == "." and item.strip(".").lower() in abbrev_to_book: 
        item = "Book of {}".format(abbrev_to_book[item.strip(".").lower()].capitalize())
    spans = refined.process_text(item)
    for s in spans: 
        ent_type = s.coarse_mention_type
        ent = s.predicted_entity
        if ent is not None: 
            new_entry = {'lemmata':item,'ent_type':ent_type,'ent_title':ent.wikipedia_entity_title,'ent_id':ent.wikidata_entity_id}
        else: 
            new_entry = {'lemmata':item,'ent_type':ent_type,'ent_title':'','ent_id':''}
        if item not in linked: 
            linked[item] = new_entry 
            linked[item]['frequency'] = 0 
        linked[item]['frequency'] += entry['frequency']
    if idx == 50: 
        break
linked = pd.DataFrame(list(linked.values()))
linked  

0it [00:00, ?it/s]/Users/amycweng/anaconda3/envs/sermons_app/lib/python3.11/site-packages/torch/amp/autocast_mode.py:250: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(
50it [00:02, 20.36it/s]


,lemmata,ent_type,ent_title,ent_id,frequency
0,God,None,God,Q190,96760
1,Christ,PERSON,Jesus,Q302,38417
2,Book of Corinthians,None,Epistles to the Corinthians,Q534518,8291
3,Book of Romans,None,Epistle to the Romans,Q48203,8095
4,Book of Psalms,None,Psalms,Q41064,7312
5,Paul,None,None,None,5746
6,Moses,None,None,None,4897
7,John,PERSON,None,None,7828
8,Book of Hebrews,None,Epistle to the Hebrews,Q128608,4033
9,Mat,None,Tatami,Q318763,2739
